# Walkthrough: from WinRiver ASCII files to the platinum matrix

Same steps as `scripts/run_pipeline.py`, one at a time. The data are not part of the repository:
put your ASCII files in `data/circuit/` (or change the paths below). This example uses a closed
circuit split into six transects; for a single straight line with the repetition times in a file,
see the comments in the repetitions cell.

In [ ]:
import sys, os
sys.path.insert(0, '..')                       # to import the package without installing it
import numpy as np
import towed_adcp as ta
from towed_adcp import plotting as pl

pl.set_style()
os.makedirs('../output/walkthrough', exist_ok=True)

## 1. Bronze matrix

Read the files, keep the usable bins and stack everything. Percent good is turned off here because
this export reports 0 in every bin; set `pg_min=70` when the column is populated.

In [ ]:
bronze, meta = ta.build_bronze('../data/circuit/*_ASC.TXT', err_max=10, q_max=100, vbt_min=10, pg_min=None)
time_base = meta['time_base']                  # (year, month) origin of the decimal day
profiles = ta.profiles_from_bronze(bronze)     # one row per profile
coast = pl.load_coast('../data/circuit/coast.shp')   # optional; None if the file is missing

pl.plot_trajectory(profiles, time_base, coast)
pl.plot_bronze_series(bronze, time_base)
pl.plot_bronze_profiles(bronze);

## 2. Repetitions

Two options. For a repeated circuit, give the turning points and the order of the legs. For a
single line with known times, read them from a file:

```python
transects = {'line': {'origin': (lon0, lat0)}}
reps = ta.repetitions_from_file('../data/line/timtran.mat', transect='line')
```

In [ ]:
vertices = {'SW': (-116.62278, 31.76041), 'SE': (-116.62034, 31.76074), 'NW': (-116.62124, 31.76778),
            'NE': (-116.61951, 31.76699), 'CW': (-116.62245, 31.76505), 'CE': (-116.62008, 31.76436)}
transects = {'South': ['SW', 'SE'], 'South_North': ['SE', 'NW'], 'North': ['NW', 'NE'],
             'North_Center': ['NE', 'CW'], 'Center': ['CW', 'CE'], 'Center_South': ['CE', 'SW']}

reps = ta.repetitions_by_vertices(profiles, vertices, transects, r_max=100, dur_max=20, gap_max=60)
geometry = ta.transect_geometry(transects, vertices)
pl.plot_repetitions(profiles, reps, time_base, vertices, coast);

## 3. Silver matrix

One Joyce correction per repetition. Check that α stays within ±0.2 rad and β within ±0.03.

In [ ]:
silver, reps = ta.build_silver(bronze, reps)
pl.plot_alpha_beta(reps, time_base)
pl.plot_correction(bronze, reps, 'North', time_base);

## 4. Gold matrices

Grid spacing: 10 m along the transect (the boat moves about 9 m between profiles) and 0.25 m in
the vertical (0.2-m bins).

In [ ]:
gold = ta.build_gold(silver, reps, geometry, dy=10, dz=0.25, z_min=0.5, z_max=7)
pl.plot_gold_panel(gold['South_North'], 'South_North', time_base, field='v');

## 5. Platinum matrices

Residual plus diurnal, semidiurnal and quarter-diurnal bands. With only 25 hours of data the
diurnal band is barely resolved; the fit at one node shows what each band adds.

In [ ]:
periods = {'D1': 23.93, 'D2': 12.42, 'D4': 6.21}
platinum = ta.build_platinum(gold, periods, n_min=25)

pl.plot_node_fit(gold['South_North'], 'South_North', periods, time_base, z=1.5, d=400)
for name in gold:
    pl.plot_platinum_panel(gold[name], platinum[name], name, time_base)
pl.plot_residual_map(gold, platinum, profiles, coast);

## 6. Along- and cross-channel components

Rotate the gold matrices to the principal axis of the depth-averaged current and fit again.
After the rotation, `u` is the along-channel component (positive in the direction of the axis,
which points north-ish) and `v` the cross-channel component (positive to the right of the axis).

In [ ]:
theta, frac = ta.principal_axis(silver)
print(f'channel axis: heading {(90 - np.rad2deg(theta)) % 360:.1f} deg, {100*frac:.0f} % of the variance')
gold_ch = ta.rotate_gold(gold, theta)
platinum_ch = ta.build_platinum(gold_ch, periods, n_min=25)

pl.plot_residual_sections(gold_ch, platinum_ch, 'u', 'Along-channel residual (cm/s)')
pl.plot_residual_sections(gold_ch, platinum_ch, 'v', 'Cross-channel residual (cm/s)', zero_line=False);

## 7. Surface temperature

Transducer temperature averaged over 4-hour windows, on the map.

In [ ]:
pl.plot_surface_temperature_maps(profiles, time_base, reps, gold, window_h=4, coast=coast);

## 8. Save

In [ ]:
out = '../output/walkthrough'
ta.save_matrix(f'{out}/bronze.npz', bronze, time_base=time_base)
ta.save_matrix(f'{out}/silver.npz', silver, time_base=time_base)
reps.to_csv(f'{out}/repetitions.csv', index=False)
ta.save_gold(f'{out}/gold.npz', gold)
ta.save_platinum(f'{out}/platinum.npz', platinum)
ta.save_platinum(f'{out}/platinum_channel.npz', platinum_ch)